In [1]:
import numpy as np
import pandas as pd 
import json
from shapely.geometry import Point, LineString, MultiLineString
import geopandas

In [2]:
colors_d = {
    'Ligne 1': '#FFCD00', 
    'Ligne 2': '#003CA6', 
    'Ligne 3': '#837902', 
    'Ligne 4': '#CF009E', 
    'Ligne 5': '#FF7E2E', 
    'Ligne 6': '#6ECA97',
    'Ligne 7': '#FA9ABA', 
    'Ligne 8': '#E19BDF',
    'Ligne 9': '#B6BD00',
    'Ligne 10': '#C9910D',
    'Ligne 11': '#704B1C',
    'Ligne 12': '#007852',
    'Ligne 13': '#6EC4E8',
    'Ligne 14': '#62259D',
    'Ligne 3bis': '#6EC4E8',
    'Ligne 7bis': '#6ECA97',
    'Couloirs': '#777779'
}

# Read data

In [3]:
file_path = 'data/raw_data/'
station_file_name = 'evolution_station.csv'
lignes_file_name = 'evolution_correspondances.csv'

In [4]:
stations_evolution = pd.read_csv(file_path+station_file_name)
correspondances_evolution = pd.read_csv(file_path+lignes_file_name)

In [5]:
correspondances_evolution['Ouvert'] = correspondances_evolution['Fermeture'].map(lambda x: not type(x)==str)
stations_evolution['Ouvert'] = stations_evolution['end_date'].map(lambda x: not type(x)==str)

In [35]:
stations_ouvertes = stations_evolution[stations_evolution['Ouvert']==True].set_index('Nom de référence')
correspondances_ouvertes = correspondances_evolution[correspondances_evolution['Ouvert']==True]

# Lines to json

In [36]:
lines = correspondances_ouvertes['Ligne'].unique()

In [37]:
paths = []

for line in lines:
    df = correspondances_ouvertes[(correspondances_ouvertes['Ligne']==line)&
                                  (correspondances_ouvertes['Ouvert']==True)]
    path = None
    for i in df.index:
        de = df.loc[i, 'De']
        vers = df.loc[i, 'Vers']
        correspondance = LineString([Point([stations_ouvertes.at[de, 'longitude'], 
                                            stations_ouvertes.at[de, 'latitude']]), 
                                     Point([stations_ouvertes.at[vers,'longitude'], 
                                            stations_ouvertes.at[vers,'latitude']])
                                    ])
        if path == None: 
            path = correspondance
        else: 
            path = path.union(correspondance)
            
    paths.append({'Ligne': line, 
                  'Couleur': colors_d[line], 
                  'geometry': path}
                 )

In [38]:
geopandas.GeoDataFrame(paths).to_file("data/lignes_ouvertes.geojson", driver='GeoJSON')

# Stations to json

In [43]:
nom_ref_to_nom = stations_ouvertes['Nom'].reset_index()
correspondances_ouvertes = correspondances_ouvertes.merge(nom_ref_to_nom, left_on='De', right_on='Nom de référence')

In [46]:
de_ligne = correspondances_ouvertes[correspondances_ouvertes['Ligne']!='Couloirs'].groupby(['Nom'])['Ligne'].unique()
de_ligne = de_ligne.reset_index()
de_ligne = de_ligne.rename(columns={'Ligne': 'Lignes'})

In [45]:
nom_ref_to_nom = nom_ref_to_nom.merge(de_ligne, on='Nom')

In [52]:
stations_ouvertes['geometry'] = stations_ouvertes.apply(lambda x: Point([x['longitude'], x['latitude']]), 
                                                        axis=1)
stations_ouvertes.drop(['end_date', 'longitude', 'latitude', 'Note', 'Ouvert'], axis=1, inplace=True)
stations_ouvertes.reset_index(inplace=True)
stations_ouvertes.rename(columns={'start_date': 'Ouverture'}, inplace=True)

stations_ouvertes = stations_ouvertes.merge(nom_ref_to_nom, on=['Nom de référence', 'Nom'])
stations_ouvertes['Lignes'] = stations_ouvertes['Lignes'].map(lambda x: list(x))
stations_ouvertes['Couleur'] = stations_ouvertes['Lignes'].map(lambda x: colors_d[x[0]] if len(x)==1 else 'D8D8B9')
stations_ouvertes['Lignes'] = stations_ouvertes['Lignes'].map(lambda x: str(x))

In [53]:
geopandas.GeoDataFrame(stations_ouvertes).to_file("data/stations.geojson", driver='GeoJSON')